# Best-of-N Speed Benchmark Across Models

Benchmark Best-of-N generation speed on the same set of MATH
problems across multiple large language models.

Each model is loaded with vLLM, warmed up, timed for
`num_trials` runs, and then unloaded before the next model is
tested. This keeps the comparison focused on model-level
throughput under the same benchmark settings.

Use this notebook to compare speed across model families or model
sizes. Use `benchmark_speed_bon_quant_v1.ipynb` for the separate
quantization sweep.

## Setup

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL + 1)

import warnings
warnings.filterwarnings("ignore")


import sys
sys.path.append("..")

import gc
import statistics
import time

import torch
from vllm import LLM

from sal.config import Config

from core import bon_search_v1
from utils.load_data import load_data_hf

In [3]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets'

ds_split = "test"
ds_dir = os.path.join(base_dir, "prm800k/math_splits")

# Models to benchmark — same prompts, same config; only this varies
llm_dirs = [
    os.path.join(base_dir, "Llama3.2-1B-Instruct"),
    os.path.join(base_dir, "Llama3.2-3B-Instruct"),
    os.path.join(base_dir, "Qwen2.5-3B-Instruct"),
    os.path.join(base_dir, "Qwen2.5-7B-Instruct"),
]

In [4]:
# Best-of-N search params
config = Config()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048
config.n = 32
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

# Benchmark knobs (all in one place so the matrix is easy to vary)
level = 4                          # MATH difficulty level
MAX_QUESTIONS = 10                 # cap on questions per benchmark
num_trials = 2                     # timed runs per model
warmup = 1                         # untimed warmup runs per model
llm_gpu_memory_utilization = 0.7

In [5]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)
num_questions = min(len(dataset), MAX_QUESTIONS)
batch_of_questions = [dataset[i]['problem'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

num_questions = 10


## Helpers

In [6]:
def gpu_mem_used_gb(device=0):
    """Driver-level used GPU memory; sees both PyTorch and vLLM allocs."""
    free, total = torch.cuda.mem_get_info(device)
    return (total - free) / (1024**3)


def benchmark_model(llm_dir, config, prompts, num_trials, warmup=1):
    """Load `llm_dir` under vLLM, warm up, time `num_trials` runs of
    best_of_n_v1, then tear down. Returns (model_name, trial_times).
    """
    model_name = os.path.basename(llm_dir.rstrip('/'))
    print(f"\n=== {model_name} ===")

    # enforce_eager=True disables CUDA graphs - skips cudagraph capture cost
    # on every model load, giving more stable latency at small num_trials.
    llm = LLM(
        model=llm_dir,
        tensor_parallel_size=1,
        max_model_len=5000,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        enforce_eager=True,
        distributed_executor_backend=None,
        dtype="float16",
        seed=config.seed,
    )
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU memory used: {gpu_mem_used_gb():.2f} GB")

    # Warmup (untimed) - absorbs first-call init inside best_of_n_v1
    for w in range(warmup):
        bon_search_v1.best_of_n_v1(prompts, config, llm, 10_000 + w)

    times = []
    for trial_idx in range(num_trials):
        start = time.perf_counter()
        bon_search_v1.best_of_n_v1(prompts, config, llm, trial_idx)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
        print(
            f"  trial {trial_idx}: {elapsed:>7.2f}s total, "
            f"{elapsed / len(prompts):.4f}s/question"
        )

    del llm
    gc.collect()
    torch.cuda.empty_cache()
    return model_name, times

## Run benchmark

One model at a time; teardown between iterations frees the vLLM
engine before the next is loaded.

In [7]:
results = []
for llm_dir in llm_dirs:
    name, times = benchmark_model(
        llm_dir, config, batch_of_questions, num_trials, warmup=warmup,
    )
    results.append((name, times))


=== Llama3.2-1B-Instruct ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.37s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.37s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  GPU memory used: 22.84 GB
  trial 0:   38.00s total, 3.7998s/question
  trial 1:   38.72s total, 3.8718s/question


[rank0]:[W612 13:17:52.894871659 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== Llama3.2-3B-Instruct ===


...

[rank0]:[W612 13:31:36.027373722 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== Qwen2.5-7B-Instruct ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:05<00:16,  5.58s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:11<00:11,  5.57s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:16<00:05,  5.55s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:21<00:00,  5.45s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:21<00:00,  5.49s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ign

  GPU memory used: 22.46 GB
  trial 0:  112.95s total, 11.2945s/question
  trial 1:  118.73s total, 11.8728s/question


[rank0]:[W612 13:38:17.839451820 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


## Summary

In [8]:
print(
    f"=== Summary (level={level}, "
    f"n_questions={num_questions}, n_trials={num_trials}) ==="
)
header = (
    f"{'model':<24}{'mean s/trial':>14}{'std':>8}{'s/question':>14}"
)
print(header)
print('-' * len(header))
for name, times in results:
    mean = statistics.mean(times)
    std = statistics.stdev(times) if len(times) > 1 else 0.0
    print(
        f"{name:<24}{mean:>14.2f}{std:>8.2f}{mean/num_questions:>14.4f}"
    )

=== Summary (level=4, n_questions=10, n_trials=2) ===
model                     mean s/trial     std    s/question
------------------------------------------------------------
Llama3.2-1B-Instruct             38.36    0.51        3.8358
Llama3.2-3B-Instruct             95.09    3.50        9.5089
Qwen2.5-3B-Instruct             160.24    5.13       16.0237
Qwen2.5-7B-Instruct             115.84    4.09       11.5837
